# Model Prediction Exploration

Explore how selected saved models behave: parish residual patterns, differences
between regions, and which fitted values align with prediction errors. Choose run,
model, target, selection, and prediction role explicitly. This notebook reads saved
predictions; it does not tune or fit models. See [README.md](README.md) for the
research sequence and the limits of repeated holdout inspection.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "pipeline" / "config").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from pipeline.modeling.inspection import select_artifacts, load_inspection_predictors
from pipeline.utils.paths import load_paths, path_value, repo_data_path
from pipeline.utils.io import normalize_id

PATHS = load_paths()
RUN_ROOT = path_value(PATHS, "external_data_root") / "outputs/epvi_prediction/runs"
pd.set_option("display.max_columns", 100)


## Choose saved results

Use run IDs printed by training, or give exact run directories. A role must exist
in every selected model: `inner_oof` for development, `outer_holdout` for an
outer-evaluation run, and `final_test` for final assessment. Keep selections
separate when comparing or plotting. `None` keeps every value available.


In [ ]:
RUN_IDS = ["REPLACE_WITH_RUN_ID"]
RUN_DIRECTORIES = [RUN_ROOT / run_id for run_id in RUN_IDS]
ROLE = "inner_oof"
MODELS = None                 # e.g. ["ridge", "random_forest"]
TARGETS = None                # e.g. ["epg_cooling", "demand_heating"]
SELECTIONS = ["holdout_A"]   # e.g. all five A-E, or ["final_test"]
MODEL_IDS = None              # Optional exact content-based model ID(s)
OUTPUT_DIRECTORY = None        # e.g. Path("<chosen folder>"); saves tables and figures
FIGURE_DIRECTORY = None
if OUTPUT_DIRECTORY is not None:
    FIGURE_DIRECTORY = Path(OUTPUT_DIRECTORY) / "figures"
    FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    def save_figure(fig, name):
        fig.savefig(FIGURE_DIRECTORY / f"{name}.png", dpi=180, bbox_inches="tight")
else:
    def save_figure(fig, name):
        return None

if not RUN_IDS or any("REPLACE" in str(path) for path in RUN_DIRECTORIES):
    raise ValueError(f"Set RUN_IDS to explicit run folders under {RUN_ROOT}.")
selected = select_artifacts(RUN_DIRECTORIES, role=ROLE, models=MODELS,
    targets=TARGETS, selections=SELECTIONS, model_ids=MODEL_IDS)
models = selected["models"]
predictions = selected["predictions"]
metrics = selected["metrics"]
regional_metrics = selected["regional_metrics"]
diagnostics = selected["diagnostics"]
print("Selected model fits:", len(models), "| prediction rows:", len(predictions))
display(models[["run_id", "model_id", "model", "target", "selection", "stage", "scope"]])


## Overall prediction patterns

Observed-versus-predicted plots reveal compression and broad bias. Residual plots
show whether errors change across the prediction range. Each panel represents one
model fit, target, and selection; residuals are observed minus predicted.


In [ ]:
for plot_number, ((run_id, model_id, model, target, selection), frame) in enumerate(predictions.groupby(
        ["run_id", "model_id", "model", "target", "selection"], sort=False), start=1):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    lo = min(frame.observed.min(), frame.predicted.min())
    hi = max(frame.observed.max(), frame.predicted.max())
    axes[0].scatter(frame.observed, frame.predicted, s=16, alpha=.55)
    axes[0].plot([lo, hi], [lo, hi], color="black", linewidth=.8)
    axes[0].set(xlabel="Observed", ylabel="Predicted", title="Observed vs predicted")
    axes[1].scatter(frame.predicted, frame.residual, s=16, alpha=.55)
    axes[1].axhline(0, color="black", linewidth=.8)
    axes[1].set(xlabel="Predicted", ylabel="Observed - predicted", title="Residual pattern")
    fig.suptitle(f"{target} / {model} / {selection} / {model_id[:12]}")
    fig.tight_layout()
    save_figure(fig, f"prediction_{plot_number:03d}")
    plt.show()


## Regional behavior

Regional means show systematic underprediction (positive residual) or overprediction
(negative residual). Small-region metrics can be unstable, so inspect the row counts
alongside the errors. These summaries describe only the selected prediction role.


In [ ]:
region_keys = ["run_id", "model_id", "model", "target", "selection", "NUTS3"]
regional_summary = predictions.groupby(region_keys, dropna=False).agg(
    n_rows=("residual", "size"), observed_mean=("observed", "mean"),
    predicted_mean=("predicted", "mean"), mean_residual=("residual", "mean"),
    mae=("residual", lambda x: x.abs().mean()), rmse=("residual", lambda x: np.sqrt(np.mean(x ** 2))),
).reset_index()
display(regional_summary.sort_values(["target", "model", "selection", "rmse"],
                                     ascending=[True, True, True, False]))
for plot_number, ((run_id, model_id, target, model, selection), frame) in enumerate(regional_summary.groupby(
        ["run_id", "model_id", "target", "model", "selection"], sort=False), start=1):
    fig, ax = plt.subplots(figsize=(9, max(3, .28 * len(frame))))
    ordered = frame.sort_values("mean_residual")
    ax.barh(ordered.NUTS3, ordered.mean_residual, color="#4477AA")
    ax.axvline(0, color="black", linewidth=.8)
    ax.set(xlabel="Mean residual (observed / predicted)", title=f"{target} / {model} / {selection} / {model_id[:12]}")
    fig.tight_layout()
    save_figure(fig, f"regional_bias_{plot_number:03d}")
    plt.show()


## Fitted-model diagnostics

Ridge coefficients and forest impurity importances answer different questions and
must not be compared on one scale. These plots show the largest absolute values for
each saved fit; impurity importance is descriptive, not causal.


In [ ]:
if not diagnostics.empty:
    diagnostic_keys = ["run_id", "model_id", "model", "target", "selection", "diagnostic"]
    for plot_number, (identity, frame) in enumerate(diagnostics.groupby(diagnostic_keys, sort=False), start=1):
        top = frame.assign(magnitude=frame.value.abs()).nlargest(20, "magnitude").sort_values("value")
        fig, ax = plt.subplots(figsize=(9, max(4, .3 * len(top))))
        ax.barh(top.feature, top.value, color="#4477AA")
        ax.axvline(0, color="black", linewidth=.8)
        ax.set(xlabel=identity[-1], title=f"{identity[3]} / {identity[2]} / {identity[4]} / {identity[1][:12]}")
        fig.tight_layout()
        save_figure(fig, f"model_diagnostic_{plot_number:03d}")
        plt.show()


## Parish residual maps

Maps require the repository parish geometry. Unmatched parishes remain uncolored.
The map uses a symmetric color scale around zero so residual direction is visible.


In [ ]:
import geopandas as gpd

geometry = gpd.read_file(repo_data_path("parishes.geojson"))[["dicofre", "geometry"]].copy()
geometry["ID_norm"] = normalize_id(geometry["dicofre"])
if geometry.ID_norm.duplicated().any():
    raise ValueError("Parish geometry contains duplicate normalized IDs.")
for (run_id, model_id, model, target, selection), frame in predictions.groupby(
        ["run_id", "model_id", "model", "target", "selection"], sort=False):
    residuals = frame[["ID_norm", "residual"]]
    mapped = geometry.merge(residuals, on="ID_norm", how="left", validate="one_to_one")
    limit = mapped.residual.abs().quantile(.98)
    if not np.isfinite(limit) or limit == 0:
        limit = max(float(mapped.residual.abs().max()), 1e-9)
    ax = mapped.plot(column="residual", cmap="coolwarm", vmin=-limit, vmax=limit,
                     legend=True, figsize=(8, 9), missing_kwds={"color":"#eeeeee"})
    fig = ax.figure
    ax.set_axis_off()
    ax.set_title(f"{target} / {model} / {selection} / {ROLE} residuals")
    plt.tight_layout()
    save_figure(fig, f"residual_map_{model}_{target}_{selection}_{model_id[:12]}")
    plt.show()


## Predictor and residual relationships

Optionally join original predictor values for one run. The helper checks that the
current modeling table matches the saved run before joining; this prevents revised
inputs from being presented alongside old predictions. Correlations are descriptive
screening only and do not show that a predictor caused the error.


In [ ]:
INSPECT_PREDICTORS = False
PREDICTOR_RUN = RUN_DIRECTORIES[0]
if INSPECT_PREDICTORS:
    predictor_values = load_inspection_predictors(PREDICTOR_RUN)
    joined = predictions.merge(predictor_values, on="ID_norm", how="left", validate="many_to_one")
    rows = []
    for (run_id, model_id, model, target, selection), frame in joined.groupby(
            ["run_id", "model_id", "model", "target", "selection"], sort=False):
        for predictor in predictor_values.columns.drop("ID_norm"):
            usable = frame[[predictor, "residual"]].dropna()
            if len(usable) >= 3 and usable[predictor].nunique() > 1 and usable.residual.nunique() > 1:
                rows.append({"run_id":run_id, "model_id":model_id, "model":model, "target":target, "selection":selection,
                             "predictor":predictor, "n":len(usable),
                             "spearman":usable.corr(method="spearman").iloc[0, 1]})
    feature_residual = pd.DataFrame(rows)
    display(feature_residual.assign(abs_rho=feature_residual.spearman.abs())
            .sort_values("abs_rho", ascending=False).groupby(
                ["run_id", "model_id", "model", "target", "selection"], sort=False).head(15))


## Export selected results

Choose a directory to save the filtered predictions, metrics, regional metrics,
model index, and figures shown above. Exports keep run IDs, model IDs, selections,
targets, and role so rows remain traceable. Leave the directory unset to inspect
without writing files.


In [ ]:
if OUTPUT_DIRECTORY is not None:
    output = Path(OUTPUT_DIRECTORY)
    output.mkdir(parents=True, exist_ok=True)
    for name, table in [("models", models), ("predictions", predictions),
                        ("metrics", metrics), ("regional_summary", regional_summary),
                        ("diagnostics", diagnostics)]:
        table.to_csv(output / f"{name}.csv", index=False)
    print("Saved selected tables and figures to", output.resolve())
